# 전체 데이터 시계열 분석 · jinny

**범위:** CSV 7개를 모두 사용한다. 온도 파일의 **모든 22개 센서·14 LOT**, 전체 라인 사건의 월별 분포, CAL01의 정비·교체·계획수리와 센서 관측을 분석한다. 마스터는 시계열이 아닌 연결·설명 자료다.

| 순서 | 분석 | 볼 것 |
| --- | --- | --- |
| 1 | [준비·연결](#prepare) | 파일 규모·키·시각·미매칭 |
| 2 | [관측 시간 지도](#timeline) | LOT·이벤트·실제 정비·계획수리의 시간 위치 |
| 3 | [전체 센서 추세](#signals) | LOT별 22채널 추세·결측·초반/후반 차이 |
| 4 | [TC–OP 관계](#relations) | 8개 존의 동시·지연 변화율 상관 |
| 5 | [사건 이력](#events) | 전체 라인 월별 분포와 CAL01 작업·교체 |
| 6 | [사건 전후 비교](#before-after) | 전후 관측이 실제 존재하는 사례만 비교 |
| 7 | [최종 결과](#results) | 근거표·그래프·HTML 보고서 |

**현재 자료 전체를 사용하는 사후 탐색**이다. 이전 분할의 시험 LOT도 포함하므로 독립 시험 성능을 주장하지 않는다. 신호 변화·상관·정비 인접성은 고장 원인의 증거가 아니다. 기존 [파생변수·모델링 전달](04_variable_summary_jinny.ipynb#timeseries-handoff)은 유지한다.

**결과 보기:** [그래프·표 해설 PDF](../reports/annealing_timeseries_explained_jinny.pdf) · [전체 LOT 그래프 HTML](../outputs/all_timeseries_jinny/report.html)

<a id="prepare"></a>
## 1. 준비와 데이터 연결

출력은 로컬 `outputs/all_timeseries_jinny/`에 저장한다. 생성된 그림·표를 모은 `report.html`에서 전체 결과를 볼 수 있다. 반복 실행 시 같은 생성 파일을 갱신한다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")       # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")             # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"                              # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display
import json
import hashlib

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "all_timeseries_jinny"
OUT.mkdir(parents=True, exist_ok=True)
fonts = {f.name for f in font_manager.fontManager.ttflist}
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font in fonts:
        rcParams["font.family"] = font
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

files = {"온도": "T-CR1-CAL01_온도.csv", "이벤트": "G-02_조업이벤트.csv",
         "계획수리": "G-01_정기수리캘린더.csv", "설비": "G-01_설비마스터.csv",
         "부품": "G-01_부품마스터.csv", "정비": "G-02_정비이력.csv", "교체": "G-02_부품교체이력.csv"}
tables = {}
for name, filename in files.items(): tables[name] = pd.read_csv(DATA / filename, encoding="utf-8-sig")
temp = tables["온도"].copy()
event = tables["이벤트"].copy()
plan = tables["계획수리"].copy()
eqp = tables["설비"].copy()
parts = tables["부품"].copy()
maint = tables["정비"].copy()
change = tables["교체"].copy()
TARGET = "P1-CR1-CAL01"
CASE_LOT = 240061  # 화면에서 자세히 볼 LOT. 다른 번호로 바꿔 실행할 수 있습니다.
signals = temp.columns.drop(["MEAS_DT", "LOT_NO"]).tolist()
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (plan, ["STRT_DT", "END_DT"]), (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column], errors="raise")
        assert frame[column].notna().all()
assert temp.LOT_NO.notna().all()
assert not temp.duplicated(["LOT_NO", "MEAS_DT"]).any()
for frame, key in [(eqp,"EQP_TAG"), (parts,"PART_CD"), (maint,"WO_NO")]:
    assert frame[key].notna().all() and frame[key].is_unique
assert (maint.STRT_DT <= maint.END_DT).all()
assert change.WO_NO.isin(maint.WO_NO).all()
assert change.PART_CD.isin(parts.PART_CD).all()
assert (eqp.EQP_TAG == TARGET).sum() == 1
cal_maint = maint[maint.EQP_TAG == TARGET].copy()
linked = change.merge(maint, on="WO_NO", validate="m:1").merge(parts, on="PART_CD", validate="m:1")
assert len(linked) == len(change)
cal_parts = linked[linked.EQP_TAG == TARGET].copy()
cr1_event = event[(event.PLANT_CD == "CR1") & (event.EVT_TYPE == "강종변경")].copy()
cr1_plan = plan[plan.PLANT_CD == "CR1"].copy()
temp = temp.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
temp["dt_s"] = temp.groupby("LOT_NO").MEAS_DT.diff().dt.total_seconds()
temp["segment_id"] = (temp.LOT_NO.ne(temp.LOT_NO.shift()) | temp.dt_s.gt(10)).cumsum()
spans = temp.groupby("LOT_NO").MEAS_DT.agg(start="min", end="max", rows="size")
grade_map = {}
for lot, row in spans.iterrows():
    exact = cr1_event[cr1_event.EVT_DT == row.start]
    grade_map[lot] = exact.AFT_VAL.iloc[0] if len(exact) == 1 else "미확인"
temp["grade_candidate"] = temp.LOT_NO.map(grade_map)

# 아래 두 함수는 표·그림을 화면과 보고서에 함께 넣기 위한 도우미입니다.
report_tables = []
report_figures = []
def show_table(name, frame):
    display(frame.head(20))
    if len(frame) > 20: print(f"전체 {len(frame)}행 중 20행 표시. 전체 값은 CSV와 HTML에서 확인합니다.")
    frame.to_csv(OUT / (name + ".csv"), index=True, encoding="utf-8-sig")
    report_tables.append((name, frame.copy()))
def show_figure(name, fig, show=True):
    fig.tight_layout()
    fig.savefig(OUT / (name + ".png"), bbox_inches="tight")
    report_figures.append(name)
    if show: plt.show()
    plt.close(fig)

inventory = pd.DataFrame([{ "자료": name, "행": len(frame), "열": len(frame.columns),
    "결측 셀": int(frame.isna().sum().sum()), "완전중복 추가행": int(frame.duplicated().sum())}
    for name, frame in tables.items()])
show_table("01_파일별_점검", inventory)

<a id="timeline"></a>
## 2. 관측 시간 지도

**이유:** 1년 범위의 데이터라도 연속 수집이 아니다. 관측이 없는 곳에서 추세·고장 전조를 추정하지 않는다. 계획수리 종료일은 해당 날짜 전체를 포함하는 가정이다. 정비의 감지·시작·완료는 다른 사건 시각이다.

| 목적 | 그림에서 볼 것 | 활용·한계 |
| --- | --- | --- |
| 사건과 센서가 같은 시기에 있는지 확인 | 파란 관측 구간과 사건의 시간 위치 | 관측이 없는 사건은 전조 비교에서 제외. 공백을 정지로 단정하지 않음 |

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
for _, row in spans.iterrows(): ax.plot([row.start, row.end], [4, 4], linewidth=8, color="steelblue")
ax.scatter(cr1_event.EVT_DT, np.full(len(cr1_event), 3), s=12, color="green")
ax.scatter(cal_maint.DETC_DT, np.full(len(cal_maint), 2), s=15, color="red")
for _, row in cal_maint.iterrows(): ax.plot([row.STRT_DT, row.END_DT], [1, 1], linewidth=5, color="orange")
for _, row in cr1_plan.iterrows(): ax.plot([row.STRT_DT, row.END_DT + pd.Timedelta(days=1)], [0, 0], linewidth=8, color="gray")
ax.set_yticks([0,1,2,3,4], ["계획수리", "실제 작업기간", "감지시각", "CR1 강종 이벤트", "온도 관측"])
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_title("CAL01 시간 지도: 사건이 있어도 센서가 없을 수 있음")
ax.grid(axis="x", alpha=0.2)
show_figure("02_관측과_사건_시간지도", fig)
show_table("02_LOT_관측범위", spans.assign(grade_candidate=spans.index.map(grade_map)))

<a id="signals"></a>
## 3. 모든 센서·모든 LOT의 추세

**표:** 센서별 유효 수·결측률·평균·변동성·실제 시간 대비 변화율·첫/마지막 10분 차이를 계산한다. 초반과 후반 차이는 추세의 단서이며 고장 점수가 아니다.

**그림:** 읽기 쉽게 같은 연속구간 안에서만 30초 단위 평균으로 표시한다. 원본을 보간하지 않으며 평균 그림이 짧은 급변을 숨길 수 있으므로 급변 검토에는 원본과 변화율 표를 사용한다. CP 두 컬럼은 척도가 달라 각각 그린다.

| 목적 | 읽는 순서 | 활용·한계 |
| --- | --- | --- |
| 전체 센서에서 관측 부족·수준 변화를 찾기 | 결측 지도 → 선택 LOT 추세 → 센서별 초반/후반 표 | 볼 센서와 구간을 선정. 평균 변화만으로 고장 판정하지 않음 |

**화면에는 `CASE_LOT` 한 개의 추세만 표시한다.** 14개 LOT의 전체 그림은 HTML에서 원하는 LOT만 펼쳐 볼 수 있다. 각 패널은 같은 신호군이며 CP는 각각 다른 축이다.

In [ ]:
sensor_rows = []
rate = temp.groupby("segment_id")[signals].diff().div(temp.dt_s, axis=0)
for lot, rows in temp.groupby("LOT_NO"):
    first = rows.MEAS_DT < rows.MEAS_DT.min() + pd.Timedelta(minutes=10)
    last = rows.MEAS_DT > rows.MEAS_DT.max() - pd.Timedelta(minutes=10)
    for sensor in signals:
        values = rows[sensor]
        sensor_rows.append({"LOT_NO": lot, "강종후보": grade_map[lot], "센서": sensor,
            "관측행": len(rows), "유효행": values.count(), "결측률(%)": values.isna().mean()*100,
            "평균": values.mean(), "표준편차": values.std(), "최소": values.min(), "최대": values.max(),
            "첫10분유효행": values[first].count(), "마지막10분유효행": values[last].count(),
            "첫10분평균": values[first].mean(), "마지막10분평균": values[last].mean(),
            "후반-초반": values[last].mean()-values[first].mean(),
            "절대변화율99백분위": rate.loc[rows.index, sensor].abs().quantile(0.99)})
sensor_stats = pd.DataFrame(sensor_rows)
show_table("03_전체센서_LOT통계", sensor_stats)

missing_grid = sensor_stats.pivot(index="센서", columns="LOT_NO", values="결측률(%)")
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(missing_grid, ax=ax, cmap="YlOrRd", annot=True, fmt=".2f",
            mask=missing_grid.eq(0), vmin=0, vmax=max(1, missing_grid.max().max()),
            cbar_kws={"label": "결측률(%)"}, annot_kws={"fontsize": 7})
ax.set_title("센서 × LOT 결측률: 흰 칸은 0%, 숫자가 있는 칸부터 점검")
show_figure("03_전체센서_결측지도", fig)

families = [(["CLN-IN", "TC-Z1", "TC-Z2"], "입측·Z1~Z2"),
            (["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"], "Z3~Z6"),
            (["TC-Z7", "TC-Z8", "TC-Z9", "TC-OUT1", "TC-OUT2"], "Z7~Z9·출측"),
            (["OP-Z1", "OP-Z2", "OP-Z3", "OP-Z4"], "OP-Z1~Z4"),
            (["OP-Z5", "OP-Z6", "OP-Z7", "OP-Z8"], "OP-Z5~Z8"),
            (["CP-Z4"], "CP-Z4"), (["CP-Z4M"], "CP-Z4M")]
assert set(signals) == {s for group, _ in families for s in group}
minute_data = []
for lot, rows in temp.groupby("LOT_NO"):
    fig, axes = plt.subplots(7, 1, figsize=(13, 17), sharex=True)
    for segment_id, segment in rows.groupby("segment_id"):
        # 표시용 30초 집계. 연속구간을 넘지 않습니다.
        sampled = segment.set_index("MEAS_DT")[signals].resample("30s", origin=segment.MEAS_DT.iloc[0]).mean()
        sampled["LOT_NO"] = lot
        sampled["segment_id"] = segment_id
        minute_data.append(sampled.reset_index())
        elapsed = (sampled.index - rows.MEAS_DT.min()).total_seconds()/60
        for ax, (group, title) in zip(axes, families):
            for color_index, sensor in enumerate(group):
                ax.plot(elapsed, sampled[sensor], label=sensor, linewidth=0.8, color=f"C{color_index}")
            ax.set_ylabel(title + "\n원본 단위")
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), loc="best", ncol=3, fontsize=8)
        ax.grid(alpha=0.2)
    axes[-1].set_xlabel("LOT 관측 시작 후 경과분")
    axes[0].set_title(f"LOT {lot} / {grade_map[lot]} / 전체 22센서 (표시용 30초 평균)")
    show_figure("03_추세_LOT_"+str(lot), fig, show=(lot == CASE_LOT))
display_samples = pd.concat(minute_data, ignore_index=True)
display_samples.to_csv(OUT / "03_표시용30초평균.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 결과는 원본에서 계산한 값으로 문장과 표를 함께 만듭니다.
focus = sensor_stats[(sensor_stats["LOT_NO"] == CASE_LOT) &
                     sensor_stats["센서"].isin(["TC-Z3", "OP-Z3", "TC-Z4", "CP-Z4", "CP-Z4M"])]
show_table("03_선택LOT_해석근거", focus[["센서", "유효행", "결측률(%)", "첫10분평균", "마지막10분평균", "후반-초반"]])
highest_missing = sensor_stats.loc[sensor_stats["결측률(%)"].idxmax()]
print(f"결과 ⇒ 결측률 최대 조합: LOT {highest_missing['LOT_NO']} / {highest_missing['센서']} / {highest_missing['결측률(%)']:.2f}%")
print("활용 ⇒ 이 구간은 신호 관계를 해석하기 전에 관측 부족부터 확인합니다.")
print("한계 ⇒ 첫·마지막 평균 차이는 관측 구간 차이이며 강종변경 효과가 아닙니다.")

<a id="relations"></a>
## 4. 8개 존의 TC–OP 동시·시간 지연 관계

OP가 있는 Z1~Z8을 모두 비교한다. Z9·출측·CP는 대응 OP가 없어 동일한 쌍 분석에서 제외한다. 같은 LOT·세그먼트 안에서 정확한 과거 시각을 찾아 **과거 OP 변화율 ↔ 현재 TC 변화율**의 상관을 계산한다.

0·10·30·60·120초 후보, 유효 쌍 100개 이상·양쪽 비상수인 경우만 표시한다. 100행은 독립 표본 100개가 아니다. 전체 LOT를 탐색하며 이 결과로 시험 성능이나 최적 지연을 주장하지 않는다.

| 목적 | 그림에서 볼 것 | 활용·한계 |
| --- | --- | --- |
| 어느 존·지연에서 함께 변하는지 비교 | 요약 히트맵의 부호·크기 → Z3 LOT별 반복성 | 상관이 큰 후보를 검토하되 최적 지연·원인으로 확정하지 않음 |

In [ ]:
lag_rows = []
for lot, rows in temp.groupby("LOT_NO"):
    for zone in range(1, 9):
        tc, op = f"TC-Z{zone}", f"OP-Z{zone}"
        for lag in [0, 10, 30, 60, 120]:
            paired_segments = []
            for _, segment in rows.groupby("segment_id"):
                current = pd.Series(rate.loc[segment.index, tc].to_numpy(), index=segment.MEAS_DT)
                output = pd.Series(rate.loc[segment.index, op].to_numpy(), index=segment.MEAS_DT)
                past = output.reindex(current.index - pd.Timedelta(seconds=lag)).to_numpy()
                paired_segments.append(pd.DataFrame({"op": past, "tc": current.to_numpy()}))
            pairs = pd.concat(paired_segments, ignore_index=True).dropna()
            corr = np.nan
            if len(pairs)>=100 and pairs.op.nunique()>1 and pairs.tc.nunique()>1:
                corr = pairs.op.corr(pairs.tc)
            lag_rows.append({"LOT_NO":lot, "존":zone, "지연초":lag, "유효쌍":len(pairs), "상관":corr})
lag_results = pd.DataFrame(lag_rows)
show_table("04_8개존_지연관계", lag_results.round(4))
# 상세 560행은 CSV로 남기고, 화면에서는 존별 LOT 중앙값을 먼저 봅니다.
lag_overview = lag_results.groupby(["존", "지연초"])["상관"].median().unstack()
show_table("04_존별_상관중앙값", lag_overview)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(lag_overview, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "LOT별 상관의 중앙값"})
ax.set_title("어느 존에서 관계가 보이나? 숫자의 부호·크기를 먼저 확인")
ax.set_xlabel("출력이 앞선 시간(초)")
ax.set_ylabel("존 번호")
show_figure("04_8개존_지연관계", fig)

# 요약에 가려지는 LOT 차이는 선택한 존에서 확인합니다.
DETAIL_ZONE = 3
detail = lag_results[lag_results["존"] == DETAIL_ZONE].pivot(index="LOT_NO", columns="지연초", values="상관")
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(detail, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title(f"Z{DETAIL_ZONE}: 관계가 여러 LOT에서 반복되는가?")
ax.set_xlabel("출력이 앞선 시간(초)")
show_figure("04_선택존_LOT별관계", fig)


In [ ]:
same_time = lag_overview[0]
print("결과 ⇒ 존별 동시 변화율 상관 중앙값")
display(same_time.rename("동시상관중앙값").to_frame().round(3))
print("활용 ⇒ 관계가 큰 존은 LOT별 상세 그림에서 반복성을 확인합니다.")
print("한계 ⇒ 음의 상관은 반대 방향 움직임이며, 출력이 온도 하락의 원인이라는 뜻은 아닙니다.")

<a id="events"></a>
## 5. 사건 이력의 시간 분포

전체 라인 분포는 자료 범위를 파악하는 용도이며 CAL01 온도와 다른 라인 사건을 직접 연결하지 않는다. 월별 건수는 가동시간으로 나눈 고장률이 아니다. 정비는 감지시각, 교체는 작업 완료시각 기준이다. 계획수리는 실제 수행 여부가 아니다.

| 목적 | 그림에서 볼 것 | 활용·한계 |
| --- | --- | --- |
| 사건 기록이 어느 달·유형에 있는지 파악 | 전체 라인과 CAL01을 분리해 보기 | 센서 관측과 대조할 달을 선정. 정비 건수는 고장률이 아님 |

In [ ]:
event_month = event.assign(month=event.EVT_DT.dt.to_period("M").astype(str))
event_month = pd.crosstab(event_month.month, event_month.PLANT_CD)
maint_month = maint.assign(month=maint.DETC_DT.dt.to_period("M").astype(str))
maint_month = pd.crosstab(maint_month.month, maint_month.MNT_TYPE)
cal_month = cal_maint.assign(month=cal_maint.DETC_DT.dt.to_period("M").astype(str))
cal_month = pd.crosstab(cal_month.month, cal_month.MNT_TYPE).reindex(maint_month.index, fill_value=0)
parts_month = cal_parts.assign(month=cal_parts.END_DT.dt.to_period("M").astype(str))
parts_month = pd.crosstab(parts_month.month, parts_month.PART_CD).reindex(maint_month.index, fill_value=0)
for name, frame in [("05_전체라인_이벤트월별",event_month), ("05_전체설비_정비월별",maint_month),
                    ("05_CAL01_정비월별",cal_month), ("05_CAL01_교체월별",parts_month)]:
    show_table(name,frame)
fig, axes=plt.subplots(2,2,figsize=(15,9))
for ax, frame, title in zip(axes.flat,[event_month,maint_month,cal_month,parts_month],
                           ["전체 라인 조업이벤트","전체 설비 감지 기록","CAL01 감지 기록","CAL01 부품 교체 내역 행"]):
    frame.plot.bar(stacked=True,ax=ax)
    ax.set_title(title)
    ax.set_ylabel("기록 건수")
    ax.legend(fontsize=7)
show_figure("05_사건월별분포",fig)
show_table("05_CAL01_부품기준정보", parts[parts.PART_CD.isin(cal_parts.PART_CD)])
show_table("05_CAL01_설비기준정보", eqp[eqp.EQP_TAG == TARGET])

<a id="before-after"></a>
## 6. 사건 전후에 비교할 값이 있는가?

정비 시작 전 10분과 완료 후 10분, 강종변경 전후 10분을 대조한다. 전후 온도 행이 하나라도 있다는 것은 완전한 관측이나 같은 조건을 의미하지 않는다. 정비 전후 표에는 센서별 유효 수·LOT·다른 정비 중첩을 함께 표시한다. 여러 작업이 겹치면 한 작업의 효과로 분리할 수 없다.

| 목적 | 먼저 볼 표 | 활용·한계 |
| --- | --- | --- |
| 정비 전후 비교가 가능한지 확인 | 전·후 유효 수, 다른 작업 중첩, LOT 번호 | 양쪽 관측이 있어도 중첩 작업이 있으면 개별 정비 효과를 분리할 수 없음 |

In [ ]:
event_coverage=[]
for _,row in cr1_event.iterrows():
    before=temp[(temp.MEAS_DT>=row.EVT_DT-pd.Timedelta(minutes=10)) & (temp.MEAS_DT<row.EVT_DT)]
    after=temp[(temp.MEAS_DT>=row.EVT_DT) & (temp.MEAS_DT<row.EVT_DT+pd.Timedelta(minutes=10))]
    event_coverage.append({"EVT_ID":row.EVT_ID,"전관측":len(before),"후관측":len(after)})
event_coverage=pd.DataFrame(event_coverage)
work_coverage=[]
work_comparison=[]
for _,row in cal_maint.iterrows():
    before=temp[(temp.MEAS_DT>=row.STRT_DT-pd.Timedelta(minutes=10)) & (temp.MEAS_DT<row.STRT_DT)]
    after=temp[(temp.MEAS_DT>row.END_DT) & (temp.MEAS_DT<=row.END_DT+pd.Timedelta(minutes=10))]
    prior=temp.MEAS_DT.between(row.DETC_DT-pd.Timedelta(hours=24),row.DETC_DT,inclusive="left")
    other=cal_maint[(cal_maint.WO_NO!=row.WO_NO) &
        (cal_maint.STRT_DT<=row.END_DT+pd.Timedelta(minutes=10)) &
        (cal_maint.END_DT>=row.STRT_DT-pd.Timedelta(minutes=10))]
    work_coverage.append({"WO_NO":row.WO_NO,"유형":row.MNT_TYPE,"전관측":len(before),"후관측":len(after),
                          "감지전24시간관측":int(prior.sum()),"다른작업중첩":len(other)})
    if len(before)>0 and len(after)>0:
        for sensor in signals:
            work_comparison.append({"WO_NO":row.WO_NO,"센서":sensor,"전유효":before[sensor].count(),
                "후유효":after[sensor].count(),"전평균":before[sensor].mean(),"후평균":after[sensor].mean(),
                "후-전":after[sensor].mean()-before[sensor].mean(),"다른작업중첩":len(other),
                "전LOT":str(before.LOT_NO.unique().tolist()),"후LOT":str(after.LOT_NO.unique().tolist())})
work_coverage=pd.DataFrame(work_coverage)
work_comparison=pd.DataFrame(work_comparison)
show_table("06_강종변경_관측가능성",event_coverage)
show_table("06_정비_관측가능성",work_coverage)
show_table("06_정비전후_모든센서",work_comparison)

# 실제 비교 가능한 작업은 센서와 작업기간을 같은 시간축에 표시합니다.
for wo in work_comparison.WO_NO.unique() if not work_comparison.empty else []:
    work=cal_maint[cal_maint.WO_NO==wo].iloc[0]
    window=temp[(temp.MEAS_DT>=work.STRT_DT-pd.Timedelta(minutes=10)) & (temp.MEAS_DT<=work.END_DT+pd.Timedelta(minutes=10))]
    fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=True)
    for _,segment in window.groupby("segment_id"):
        for sensor in ["TC-Z3","TC-Z4","TC-Z5","TC-Z6"]:axes[0].plot(segment.MEAS_DT,segment[sensor],label=sensor)
        for sensor in ["OP-Z3","OP-Z4"]:axes[1].plot(segment.MEAS_DT,segment[sensor],label=sensor)
    for ax in axes:
        ax.axvspan(work.STRT_DT,work.END_DT,color="orange",alpha=0.2)
        handles,labels=ax.get_legend_handles_labels();unique=dict(zip(labels,handles))
        ax.legend(unique.values(),unique.keys())
        ax.grid(alpha=0.2)
    axes[0].set_title(wo+": 주황은 작업기간, 변화의 원인으로 확정하지 않음")
    axes[0].set_ylabel("온도 원본 단위");axes[1].set_ylabel("출력 원본 단위")
    fig.autofmt_xdate()
    show_figure("06_작업전후_"+wo,fig)

In [ ]:
usable = work_coverage[(work_coverage["전관측"]>0) & (work_coverage["후관측"]>0)]
show_table("06_전후비교_최종판단", usable)
print(f"결과 ⇒ 전후 관측 작업 {len(usable)}건 / 다른 작업과 겹치는 사례 {(usable['다른작업중첩']>0).sum()}건")
print("활용 ⇒ 센서의 전후 변화는 확인 가능하지만 개별 정비 효과는 보류합니다.")

<a id="zero-output"></a>

## 6-추가. 출력 0은 어느 존·LOT에서 관측되는가?

**목적:** 산점도에서 보인 출력 0을 Z3에 한정하지 않고 OP-Z1~Z8 전체에서 확인한다. 0을 결측이나 목표온도 도달로 간주하지 않는다.

**읽는 법:** 출력 0의 행 수·관측 지속시간·해당 구간 온도 범위를 함께 본다. 연속구간 첫 행이 0이면 실제 시작은 더 이전일 수 있다. 구간 내 작은 관측 공백에도 연속 상태가 유지됐다고 가정하므로 ‘관측된 지속시간’이다.

**활용:** 출력 0 여부·관측 지속시간·동시 온도 변화율을 점검 후보 특징으로 연결한다. 장치 정지·목표 도달·고장 여부는 목표값과 운전 모드 확인 전 보류한다.


In [ ]:

zero_rows=[]
for zone in range(1,9):
    op,tc=f"OP-Z{zone}",f"TC-Z{zone}"
    is_zero=temp[op].eq(0)
    new_run=is_zero.ne(is_zero.shift()) | temp.segment_id.ne(temp.segment_id.shift())
    run_id=new_run.cumsum()
    for _,rows in temp[is_zero].groupby(run_id[is_zero]):
        zero_rows.append({"존":zone,"LOT_NO":rows.LOT_NO.iloc[0],"start":rows.MEAS_DT.min(),
            "end":rows.MEAS_DT.max(),"관측행":len(rows),
            "관측지속초":(rows.MEAS_DT.max()-rows.MEAS_DT.min()).total_seconds(),
            "온도유효행":rows[tc].count(),"온도최소":rows[tc].min(),"온도최대":rows[tc].max()})
zero_runs=pd.DataFrame(zero_rows,columns=["존","LOT_NO","start","end","관측행","관측지속초","온도유효행","온도최소","온도최대"])
show_table("06_전체존_출력0구간",zero_runs)
zero_summary=zero_runs.groupby("존").agg(구간수=("LOT_NO","size"),LOT수=("LOT_NO","nunique"),
    관측행=("관측행","sum"),최대관측지속초=("관측지속초","max")).reindex(range(1,9),fill_value=0)
show_table("06_전체존_출력0요약",zero_summary)
fig,axes=plt.subplots(1,2,figsize=(11,4))
sns.barplot(data=zero_summary.reset_index(),x="존",y="구간수",ax=axes[0],color="steelblue")
sns.barplot(data=zero_summary.reset_index(),x="존",y="최대관측지속초",ax=axes[1],color="darkorange")
axes[0].set_title("어느 존에 출력 0 관측이 있나?")
axes[1].set_title("0이 관측된 가장 긴 구간은?")
show_figure("06_출력0_존별비교",fig)
print(f"결과 ⇒ 출력 0 관측 구간 {len(zero_runs)}개. 상세표의 온도 범위를 함께 확인합니다.")
print("해석 ⇒ 출력 0은 기록값이며 목표온도 도달·고장을 뜻하지 않습니다.")


<a id="results"></a>
## 7. 최종 분석 결과와 보고서

**모든 센서와 사건을 시간축으로 확인하되**, 마스터는 설명 정보로 활용했다. 결과 요약은 아래 계산값을 따른다. 실제 고장·정상 임곗값과 목표 온도는 확인되지 않았다. 센서별 단위와 동작 조건을 확인하기 전 서로 다른 센서의 절대 변화 크기를 중요도 순위로 사용하지 않는다.

In [ ]:
# 각 센서 안에서 변화가 큰 LOT를 고릅니다. 센서끼리 크기 순위를 매기지 않습니다.
review = sensor_stats.copy()
review["초후반차이절댓값"] = review["후반-초반"].abs()
review = review.sort_values("초후반차이절댓값", ascending=False).groupby("센서", sort=False).head(1)
review = review[["센서", "LOT_NO", "강종후보", "첫10분유효행", "마지막10분유효행", "첫10분평균", "마지막10분평균", "후반-초반", "결측률(%)"]]
show_table("07_센서별_우선검토LOT", review.sort_values("센서"))

zero_lag = lag_results[lag_results["지연초"]==0].groupby("존")["상관"].agg(["min","median","max"])
show_table("07_존별_동시변화율상관", zero_lag)
direction = lag_results.groupby(["존","지연초"])["상관"].median().unstack()
show_table("07_존별_지연상관중앙값", direction)

print("결론: 센서마다 같은 조건에서의 시간 변화와 결측을 확인했습니다.")
print("센서별 우선검토 LOT는 변화 위치를 고르는 목록이며 고장 순위가 아닙니다.")
print("상관은 LOT별 일관성·관측 쌍수를 함께 봐야 하며, 제어 반응이나 고장 원인은 확정할 수 없습니다.")


In [ ]:
both_events=(event_coverage["전관측"]>0)&(event_coverage["후관측"]>0)
both_work=(work_coverage["전관측"]>0)&(work_coverage["후관측"]>0)
summary=pd.DataFrame([
    ["분석 범위",f"CSV {len(files)}개 / 센서 {len(signals)}개 / LOT {len(spans)}개","파일 전체 범위 확인"],
    ["센서별 LOT 통계",str(len(sensor_stats))+"개 조합","분포·결측·변화량 확인"],
    ["연속 관측",str(temp.segment_id.nunique())+"구간","10초 초과 공백 분리 가정"],
    ["시간 지연 비교",str(len(lag_results))+"개 LOT·존·지연 조합","관측 쌍수와 상관 함께 확인"],
    ["강종 전후",str(int(both_events.sum()))+"건 양쪽 관측","10분 창, 변경 효과 판단 제한"],
    ["정비 전후",str(int(both_work.sum()))+"건 양쪽 관측","중첩 작업·조건 차이로 인과 해석 제한"],
    ["감지 전 관측",str(int((work_coverage["감지전24시간관측"]>0).sum()))+"작업","24시간 전체 관측을 의미하지 않음"],
],columns=["항목","결과","핵심 해석"])
show_table("07_최종요약",summary)

# 이번 분석이 사용한 자료를 기록합니다.
manifest={"source_sha256":{},"signals":signals,"gap_seconds":10,"plot_bin_seconds":30,
          "lags_seconds":[0,10,30,60,120],"scope":"retrospective_all_lots_not_model_validation"}
for name,filename in files.items():manifest["source_sha256"][filename]=hashlib.sha256((DATA/filename).read_bytes()).hexdigest()
(OUT/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")

# 결과와 목적을 먼저 보여주고 상세 그림은 필요한 것만 펼쳐 봅니다.
import html
captions = {
    "06_출력0": "목적: OP-Z1~Z8의 출력 0 구간을 비교. 존별 관측 지속시간과 해당 온도를 함께 확인. 목표 도달·정지·고장을 뜻하는지는 운전 정보 확인이 필요함.",
    "02_": "목적: 관측과 사건의 시간 위치 확인. 활용: 실제 센서가 있는 사건만 검토. 공백은 정지 확정이 아님.",
    "03_전체": "목적: 관측 부족 위치 확인. 숫자가 표시된 센서·LOT부터 점검. 결측은 고장 라벨이 아님.",
    "03_추세": "목적: 같은 LOT에서 22센서의 시간 변화를 비교. 30초 평균은 표시용이며 짧은 급변은 원본으로 재확인.",
    "04_": "목적: TC–OP 변화율 관계 비교. 부호·크기와 LOT 반복성을 함께 확인. 지연·인과·고장 확정은 불가.",
    "05_": "목적: 사건의 월별·유형별 분포 확인. CAL01과 전체 자료를 구분. 건수는 가동시간으로 보정한 고장률이 아님.",
    "06_": "목적: 작업 전후 관측 변화 확인. 주황은 작업기간. 다른 작업과 겹치므로 개별 작업 효과는 미확정.",
}
report=["<html><head><meta charset='utf-8'><title>전체 시계열 분석</title><style>body{font-family:Malgun Gothic,sans-serif;margin:30px;line-height:1.6}img{max-width:100%}table{border-collapse:collapse;font-size:13px}th,td{border:1px solid #ddd;padding:7px}th{background:#eef4fa}.table{overflow:auto;max-height:500px}section,details{margin:25px 0}summary{cursor:pointer;font-weight:bold}</style></head><body>"]
report.append("<h1>전체 데이터 시계열 분석</h1><p>목적 → 결과 요약 → 관심 그림 → 상세 표 순서로 확인하세요. 전체자료 탐색이며 예측 성능 평가는 아닙니다.</p>")
report.append(summary.to_html(index=False))
report.append("<h2>핵심 결론</h2><ul><li>센서·LOT별 관측 품질이 다릅니다. 결측을 먼저 확인한 뒤 신호를 비교합니다.</li><li>TC–OP 관계는 존별로 다릅니다. 같은 크기의 상관이 모든 존에 적용되지 않습니다.</li><li>강종변경 전후 10분 양쪽 관측은 없고, 정비 전후 비교 후보 2건도 다른 작업과 겹칩니다. 조건변경·정비 효과의 인과 해석은 보류합니다.</li></ul>")
report.append("<h2>그림 선택</h2><ul>")
for name in report_figures:report.append(f'<li><a href="#{name}">{html.escape(name)}</a></li>')
report.append("</ul>")
for name in report_figures:
    caption = next((value for prefix,value in captions.items() if name.startswith(prefix)), "목적과 결과는 관련 요약표를 참고하세요.")
    opened = "" if name.startswith("03_추세") else " open"
    report.append(f'<details id="{name}"{opened}><summary>{html.escape(name)}</summary><p>{html.escape(caption)}</p><img loading="lazy" src="{name}.png"></details>')
report.append("<h2>상세 결과표</h2>")
for name,frame in report_tables:
    report.append(f'<details><summary>{html.escape(name)}</summary><p>전체 표: <a href="{name}.csv">CSV 다운로드</a></p><div class="table">{frame.to_html(index=True, float_format=lambda x: f"{x:.5g}")}</div></details>')
report.append("<p>목표 온도·고장 정답은 미확인입니다. 34개 특징의 기존 모델링 전달 파일은 변경하지 않았습니다.</p></body></html>")
(OUT/"report.html").write_text("\n".join(report),encoding="utf-8")
assert len(sensor_stats)==len(signals)*temp.LOT_NO.nunique()
assert len(lag_results)==temp.LOT_NO.nunique()*8*5
print("완료:",OUT/"report.html")
